In [8]:
from pynwb import NWBFile, NWBHDF5IO, TimeSeries
from pynwb.image import ImageSeries
from scipy.io import wavfile
from pynwb.epoch import TimeIntervals

#from ndx_sound import AcousticWaveformSeries


from dateutil.tz import tzlocal 
from datetime import datetime

from pynwb.file import Subject, DynamicTable


from natsort import natsorted
import os
import numpy as np
import pandas as pd
from neuroconv.datainterfaces import SLEAPInterface
import sleap_io
import ffmpeg
import subprocess
import json


In [9]:
def get_video_info_ffprobe(video_path):
    cmd = [
        'ffprobe',
        '-v', 'error',
        '-select_streams', 'v:0',
        '-show_entries', 'stream=duration,nb_frames,r_frame_rate',
        '-of', 'json',
        video_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    info = json.loads(result.stdout)
    
    stream = info['streams'][0]
    duration = float(stream.get('duration', 0))
    frame_count = int(stream.get('nb_frames', 0))
    return duration, frame_count

In [10]:


#nwbfile_path = '/Users/thoman1/Documents/EMBER/Code/Sanes/NWB_Files/sanesData_0.nwb'
file_path = '/Users/thoman1/Documents/EMBER/Code/Sanes/Example Data/'

nwbfile_path = '/Users/thoman1/Documents/EMBER/Code/Sanes/NWB_Files/sanesData_v2.nwb'

# get all the idx_0 directories
folders = [f for f in os.listdir(file_path) if os.path.isdir(os.path.join(file_path, f)) and f.startswith('idx_')]
# sort the folders in natural order

folders = natsorted(folders)
#print(folders)
# get all the .wav files in the directory
nwbfile = NWBFile(session_description='Multi-animal social vocal interactions',
                  identifier='example_id',
                  session_start_time=datetime.now(tzlocal()),
         
                  )
audiochannel_data = []
sample_rates = []
vid_files = []
vox_data = []
pose_data = []



# registry so identical skeletons are created once and then reused


# vox_data = TimeIntervals(name='vocalizations')
# vox_data.add_column(name = 'name')
# vox_data.add_column(name = 'channel')
cumulative_offset = 0.0
vidStartFrames = []
vidStartFrames.append(cumulative_offset)
# cache metadata once per animal/track
# What we’ll copy directly onto PoseEstimation later


# loop through each folder
for folder in folders:
    folder_path = os.path.join(file_path, folder)
    wav_files = [f for f in os.listdir(folder_path) if f.endswith('.wav') and 'multichannel' not in f]
    wav_files = natsorted(wav_files)  
    # read the annotations.csv file
    
    channel_columns = []
    for wav_file in wav_files:
        this_file_path = os.path.join(folder_path, wav_file)
        #print(this_file_path)
        audio_sampling_rate, thisData = wavfile.read(this_file_path)  
        #append data as column in audiochannel_data
        thisData = thisData.reshape(-1, 1)
        # data = np.hstack((data, thisData.reshape(-1, 1)))
        channel_columns.append(thisData)

    data = np.hstack(channel_columns)
    audiochannel_data.append(data)
#print(audiochannel_data)

    vid_file = [f for f in os.listdir(folder_path) if f.endswith('.mp4')]
    vid_file = vid_file[0]
    vid_path = os.path.join(folder_path, vid_file)
    duration, frame_count = get_video_info_ffprobe(vid_path)
    print(f"Video: {vid_file}, Duration: {duration} seconds, Frame Count: {frame_count}")
    #print(os.path.join(folder_path,vid_file))
    vid_files.append(vid_path)

    # vocalizations data
    annotationsPath = os.path.join(folder_path, 'annotations.csv')
    df = pd.read_csv(annotationsPath)
    df.start_seconds = df.start_seconds + cumulative_offset
    df.stop_seconds = df.stop_seconds + cumulative_offset
    # append df to vox_data
    vox_data.append(df)
    cumulative_offset += duration
    vidStartFrames.append(frame_count + vidStartFrames[-1])
    

    # slp data
    # find single .slp file in the folder
    slp_files = [f for f in os.listdir(folder_path) if f.endswith('.slp')]
    slp_path = os.path.join(folder_path, slp_files[0])

   

    interface = SLEAPInterface(file_path=slp_path, verbose=False)
    metadata = interface.get_metadata()
    #print(metadata)
    interface.add_to_nwbfile(nwbfile=nwbfile, metadata=metadata)
    
    # pose data
    # open the nwb file associated with this folder that contains SLEAP pose data
    # find file that ends with .nwb
    # pose_nwb_files = [f for f in os.listdir(folder_path) if f.endswith('.nwb')]

    # pose_nwb_path = pose_nwb_files[0]

    # with NWBHDF5IO(os.path.join(folder_path,pose_nwb_path), 'r', load_namespaces=True) as io:
    #     pose_nwbfile = io.read()

       
               

               

   



print(vidStartFrames)  



# stack the audio data into a single array

multichannel_data = np.concatenate(audiochannel_data,axis = 0)
# stack into shape (n_smaples, n_channels)
#audio_matrix = np.stack(multichannel_data, axis=-1)


# acoustic_waveform_series = AcousticWaveformSeries( # got an error with dandi validate complaining about # channels
#     name='Multi-channel Acoustic Data',
#     description="Audio data from 12 channels",
#     data = multichannel_data,
#     rate = float(audio_sampling_rate),

# )

acoustic_waveform_series = TimeSeries(
    name='Multi-channel Acoustic Data',
    description="Audio data from 12 channels",
    data = multichannel_data,
    rate = float(audio_sampling_rate),
    starting_time = 0.0,
    unit = 'V',

)


nwbfile.add_acquisition(acoustic_waveform_series)






#externalFile = [os.path.relpath(os.path.join(file_path, 'combined_output.mp4'), os.path.dirname(nwbfile_path))]
externalFile = [os.path.relpath(f, os.path.dirname(nwbfile_path)) for f in vid_files]

print(externalFile)
sampling_rate = 30.0  # Assuming a default frame rate for the video

   
    # create an ImageSeries for each mp4 file
extBehVideos = ImageSeries(
    name="External Behavioral Videos",
    unit='n.a.',
    starting_time=0.0,
    starting_frame = vidStartFrames[:-1],
    rate=sampling_rate,
    description="External video file combined from multiple videos",
    external_file= externalFile,  # Path to the video file
    format = "external",
)
nwbfile.add_acquisition(extBehVideos)   

# add annotations data to nwbfile

#create bheavioral module
#behaviorModule = nwbfile.create_processing_module(name='Behavioral', description='Behavioral data processing module including pose and vocalizations')

# behavior modele from pose
behaviorModule = nwbfile.get_processing_module('behavior')
# Step 1: Concatenate all annotation DataFrames
all_vox_df = pd.concat(vox_data, ignore_index=True)

# Step 2: Rename time columns to match NWB
all_vox_df = all_vox_df.rename(columns={
    'start_seconds': 'start_time',
    'stop_seconds': 'stop_time',
    'name': 'label'
})

# Step 3: Create TimeIntervals from DataFrame
vocalizations_table = TimeIntervals.from_dataframe(
    name='vocalizations',
    df=all_vox_df
)


# print(all_vox_df.iloc[5000,:])
behaviorModule.add_container(vocalizations_table)





nwbfile

Video: center-session_135_video-0.mp4, Duration: 359.9964 seconds, Frame Count: 10800
Video: center-session_135_video-1.mp4, Duration: 359.99681 seconds, Frame Count: 10800
Video: center-session_135_video-2.mp4, Duration: 359.99722 seconds, Frame Count: 10800
Video: center-session_135_video-3.mp4, Duration: 359.99662 seconds, Frame Count: 10800
Video: center-session_135_video-4.mp4, Duration: 360.030363 seconds, Frame Count: 10801
Video: center-session_135_video-5.mp4, Duration: 359.99643 seconds, Frame Count: 10800
Video: center-session_135_video-6.mp4, Duration: 255.031233 seconds, Frame Count: 7651
[0.0, 10800.0, 21600.0, 32400.0, 43200.0, 54001.0, 64801.0, 72452.0]
['../Example Data/idx_0/center-session_135_video-0.mp4', '../Example Data/idx_1/center-session_135_video-1.mp4', '../Example Data/idx_2/center-session_135_video-2.mp4', '../Example Data/idx_3/center-session_135_video-3.mp4', '../Example Data/idx_4/center-session_135_video-4.mp4', '../Example Data/idx_5/center-session_135

root pynwb.file.NWBFile at 0x5046727568
Fields:
  acquisition: {
    External Behavioral Videos <class 'pynwb.image.ImageSeries'>,
    Multi-channel Acoustic Data <class 'pynwb.base.TimeSeries'>
  }
  devices: {
    camera_0 <class 'pynwb.device.Device'>
  }
  file_create_date: [datetime.datetime(2025, 9, 5, 15, 9, 41, 212186, tzinfo=tzlocal())]
  identifier: example_id
  processing: {
    SLEAP_VIDEO_000_center-session_135_video-0 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-1 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-2 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-3 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-4 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-5 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-6 <class 'pynwb.base.ProcessingModule'>,
    behavior <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Multi-animal social vocal interactions
  session_start_time: 2025-09-05 15:09:41.209427-04:00
  timestamps_reference_time: 2025-09-05 15:09:41.209427-04:00

In [ ]:
# -- only need to run this once ever to create the combined file -- 

# loop through vid_files and concatenate videos into one big video
# Step 1: Convert each input .mp4 to .ts (MPEG Transport Stream)
ts_files = []
for file in vid_files:
    ts_file = file.replace('.mp4', '.ts')
    ffmpeg.input(file).output(ts_file, format='mpegts', vcodec='copy', acodec='copy').run(overwrite_output=True)
    ts_files.append(ts_file)

# Step 2: Concatenate all .ts files into a single MP4
concat_str = '|'.join(ts_files)
print(concat_str)
ffmpeg.input(f'concat:{concat_str}', format='mpegts') \
      .output(os.path.join(file_path,'combined_output.mp4'), vcodec='copy', acodec='copy') \
      .run(overwrite_output=True)

# Optional: clean up intermediate .ts files
for ts_file in ts_files:
    os.remove(ts_file)

In [11]:
externalFile = [os.path.relpath(os.path.join(file_path, 'combined_output.mp4'), os.path.dirname(nwbfile_path))]

# loop through mp4_files and add them to ImageSeries 
print(externalFile)
sampling_rate = 30.0  # Assuming a default frame rate for the video

   
    # create an ImageSeries for each mp4 file
extBehVideos = ImageSeries(
    name="External Behavioral Video (Combined)",
    unit='n.a.',
    starting_time=0.0,
    rate=sampling_rate,
    description="External video file combined from multiple videos",
    external_file= externalFile,  # Path to the video file
    format = "external",
)
nwbfile.add_acquisition(extBehVideos)   
nwbfile

['../Example Data/combined_output.mp4']


root pynwb.file.NWBFile at 0x5046727568
Fields:
  acquisition: {
    External Behavioral Video (Combined) <class 'pynwb.image.ImageSeries'>,
    External Behavioral Videos <class 'pynwb.image.ImageSeries'>,
    Multi-channel Acoustic Data <class 'pynwb.base.TimeSeries'>
  }
  devices: {
    camera_0 <class 'pynwb.device.Device'>
  }
  file_create_date: [datetime.datetime(2025, 9, 5, 15, 9, 41, 212186, tzinfo=tzlocal())]
  identifier: example_id
  processing: {
    SLEAP_VIDEO_000_center-session_135_video-0 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-1 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-2 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-3 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-4 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-5 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-6 <class 'pynwb.base.ProcessingModule'>,
    behavior <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Multi-animal social vocal interactions
  session_start_time: 2025-09-05 15:09:41.209427-04:00
  timestamps_reference_time: 2025-09-05 15:09:41.209427-04:00

In [12]:




# add subject information
subject = Subject(
    subject_id='Gerbil_001_002_003',
    description='Multiple gerbils used in the experiment',
    species='Meriones unguiculatus', 
    age= 'P90D',
    sex = 'M',
)

nwbfile.subject = subject

subjectTable = DynamicTable(name='Subject Information', description='Information about the subject(s)')
subjectTable.add_column('subject_id', 'Unique subject ID')
subjectTable.add_column('age', 'Age')
subjectTable.add_column('species', 'Species')
subjectTable.add_column('sex','Sex')

subjectTable.add_row(subject_id='S0', age='P90D', species='Meriones unguiculatus',sex="M")
subjectTable.add_row(subject_id='S2', age='P92D', species='Meriones unguiculatus',sex ="F")
subjectTable.add_row(subject_id='S3', age='P95D', species='Meriones unguiculatus',sex = "M")

general_module = nwbfile.create_processing_module(name = 'general', description = "general metadata about multisubject")
general_module.add(subjectTable)
nwbfile

root pynwb.file.NWBFile at 0x5046727568
Fields:
  acquisition: {
    External Behavioral Video (Combined) <class 'pynwb.image.ImageSeries'>,
    External Behavioral Videos <class 'pynwb.image.ImageSeries'>,
    Multi-channel Acoustic Data <class 'pynwb.base.TimeSeries'>
  }
  devices: {
    camera_0 <class 'pynwb.device.Device'>
  }
  file_create_date: [datetime.datetime(2025, 9, 5, 15, 9, 41, 212186, tzinfo=tzlocal())]
  identifier: example_id
  processing: {
    SLEAP_VIDEO_000_center-session_135_video-0 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-1 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-2 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-3 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-4 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-5 <class 'pynwb.base.ProcessingModule'>,
    SLEAP_VIDEO_000_center-session_135_video-6 <class 'pynwb.base.ProcessingModule'>,
    behavior <class 'pynwb.base.ProcessingModule'>,
    general <class 'pynwb.base.ProcessingModule'>
  }
  session_description: Multi-animal social vocal interactions
  session_start_time: 2025-09-05 15:09:41.209427-04:00
  subject: subject pynwb.file.Subject at 0x5046730128
Fields:
  age: P90D
  age__reference: birth
  description: Multiple gerbils used in the experiment
  sex: M
  species: Meriones unguiculatus
  subject_id: Gerbil_001_002_003

  timestamps_reference_time: 2025-09-05 15:09:41.209427-04:00

In [13]:

# save the nwb file
with NWBHDF5IO(nwbfile_path, 'w') as io:
    io.write(nwbfile)
    print(f"NWB file saved to {nwbfile_path}")
    

/Users/thoman1/anaconda3/envs/nwb_neurconv_sleap/lib/python3.13/site-packages/hdmf/build/objectmapper.py:267: DtypeConversionWarning: Spec 'ImageSeries/external_file/starting_frame': Value with data type float64 is being converted to data type int64 (min specification: int32).
  warnings.warn(full_warning_msg, DtypeConversionWarning)


NWB file saved to /Users/thoman1/Documents/EMBER/Code/Sanes/NWB_Files/sanesData_v2.nwb
